# 📦 Notebook 01 — Data Inventory
Check dataset balance, image quality, and folder structure before touching any code.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import sys; sys.path.insert(0, '..')
from preprocessing.config import DATASET_RAW, IMG_EXTENSIONS, BLUR_THRESHOLD
print('✅ imports ok')

## 1 · Count images per species

In [ ]:
raw = Path(DATASET_RAW)
counts = {}
for sp_dir in sorted(raw.iterdir()):
    if sp_dir.is_dir():
        imgs = [f for f in sp_dir.iterdir() if f.suffix.lower() in IMG_EXTENSIONS]
        counts[sp_dir.name] = len(imgs)

print('='*45)
print(f'  {"Species":<25} {"Count":>8}  Status')
print('='*45)
for sp, n in counts.items():
    status = '✅' if n >= 25 else '⚠️  LOW'
    print(f'  {sp:<25} {n:>8}  {status}')
print('='*45)
print(f'  Total species : {len(counts)}')
print(f'  Total images  : {sum(counts.values())}')
print(f'  Min per class : {min(counts.values()) if counts else 0}')
print(f'  Max per class : {max(counts.values()) if counts else 0}')

## 2 · Class balance bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(counts.keys(), counts.values(), color='#2ecc71', edgecolor='#27ae60')
ax.axhline(25, color='orange', linestyle='--', label='Min recommended (25)')
ax.set_title('Class Balance — Raw Dataset', fontsize=13, fontweight='bold')
ax.set_xlabel('Species'); ax.set_ylabel('Image Count')
ax.legend()
plt.xticks(rotation=30, ha='right')
for b, n in zip(bars, counts.values()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, str(n), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3 · Blur score distribution — find bad images

In [ ]:
all_scores = []
bad_images = []

for img_path in sorted(raw.rglob('*')):
    if img_path.suffix.lower() not in IMG_EXTENSIONS:
        continue
    img = cv2.imread(str(img_path))
    if img is None: continue
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    score = cv2.Laplacian(gray, cv2.CV_64F).var()
    all_scores.append(score)
    if score < BLUR_THRESHOLD:
        bad_images.append((img_path, round(score, 1)))

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(all_scores, bins=40, color='steelblue', edgecolor='white')
ax.axvline(BLUR_THRESHOLD, color='red', linestyle='--', label=f'Threshold={BLUR_THRESHOLD}')
ax.set_title('Blur Score Distribution'); ax.set_xlabel('Laplacian Variance'); ax.legend()
plt.tight_layout(); plt.show()

print(f'Total images scanned : {len(all_scores)}')
print(f'Would be REJECTED    : {len(bad_images)}')
if bad_images:
    print('\nBlurry images (review these):')
    for p, s in bad_images:
        print(f'  score={s:6.1f}  {p}')

## 4 · Sample grid — one image per species

In [ ]:
species_list = [d for d in raw.iterdir() if d.is_dir()]
n = len(species_list)
if n == 0:
    print('No species folders found in', raw)
else:
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1: axes = [axes]
    for ax, sp_dir in zip(axes, species_list):
        imgs = list(sp_dir.glob('*.jpg')) + list(sp_dir.glob('*.png'))
        if imgs:
            img = cv2.imread(str(imgs[0]))
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(sp_dir.name, fontsize=8); ax.axis('off')
    plt.suptitle('One sample per species (raw)', fontweight='bold')
    plt.tight_layout(); plt.show()

## 5 · Image resolution check

In [ ]:
resolutions = Counter()
for img_path in raw.rglob('*'):
    if img_path.suffix.lower() not in IMG_EXTENSIONS: continue
    img = cv2.imread(str(img_path))
    if img is not None:
        resolutions[img.shape[:2]] += 1
print('Image resolutions found (H×W):')
for res, cnt in resolutions.most_common():
    print(f'  {res[0]}×{res[1]}  →  {cnt} images')